# EDA DeepWeeds — fold 0
**2A202602810 · Nguyễn Thái Anh**

Notebook chạy trên CPU: kiểm tra nguồn CSV/split, phân bố lớp, chất lượng ảnh, duplicate byte,
thống kê màu từ train và ảnh mẫu/augmentation. Không huấn luyện hoặc đánh giá model trên test.

Ảnh trực quan và thống kê pixel chỉ dùng **train**. Metadata train/val/test được dùng để kiểm tra
split. Kiểm tra hash ảnh các tập chỉ là audit trùng dữ liệu, không dùng để chọn siêu tham số.

Chọn kernel **.venv/bin/python** trong IDE, rồi Run All. Từ gốc repo hoặc thư mục code, notebook tự tìm
ROOT; nếu đặt notebook ở nơi khác, sửa ROOT_HINT ở cell đầu.
Kết quả: **curves/eda/** (hình) và **code/validation/eda/** (bảng/JSON/ghi chú).

## 1. Môi trường, đường dẫn và seed

In [ ]:
import os, sys, json, hashlib, random, platform
from pathlib import Path
from collections import defaultdict
import importlib.metadata

ROOT_HINT = None  # ví dụ: "/đường/dẫn/tới/repo"
if ROOT_HINT is not None:
    ROOT = Path(ROOT_HINT).expanduser().resolve()
else:
    ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                 if (p / "eval.py").is_file() and (p / "starter").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Đặt ROOT_HINT tới gốc repo có eval.py và starter/")
SUBMISSION = ROOT / "submissions/2A202602810_NguyenThaiAnh"
CODE = SUBMISSION / "code"
IMAGES_DIR, LABELS_DIR = ROOT / "data", ROOT / "data/labels"
FIG_DIR, TABLE_DIR = SUBMISSION / "curves/eda", CODE / "validation/eda"
for p in (FIG_DIR, TABLE_DIR, ROOT / ".cache/matplotlib"):
    p.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".cache/matplotlib"))
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(CODE))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
import dataset, losses
from train import write_json, _verify_provenance
from eval import compute_metrics

get_ipython().run_line_magic("matplotlib", "inline")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(4)
plt.rcParams.update({"figure.dpi": 110, "font.size": 10})
pd.set_option("display.max_columns", 20)

environment = {
    "python": platform.python_version(), "seed": SEED,
    "versions": {name: importlib.metadata.version(name)
                 for name in ("numpy", "pandas", "Pillow", "matplotlib", "torch", "torchvision")},
    "dataset_module": str(Path(dataset.__file__).resolve()),
    "image_directory": str(IMAGES_DIR),
}
write_json(TABLE_DIR / "environment.json", environment)
display(environment)

## 2. Kiểm tra nguồn dữ liệu và split
Split gốc chỉ có **Filename,Label**. Species được bổ sung trong bộ nhớ từ mapping lớp ở labels.csv.
Các CSV giữ nguyên byte; hash được đối chiếu với bản tải từ tác giả đã lưu.

Audit dừng nếu giao khác rỗng, sai union, thiếu ảnh hoặc tỉ lệ không đúng yêu cầu.
Sai khác giữa nhãn split và labels.csv được ghi lại; **nhãn split gốc là nhãn dùng cho lab**.

In [ ]:
required = ["labels.csv", "train_subset0.csv", "val_subset0.csv", "test_subset0.csv"]
missing = [n for n in required if not (LABELS_DIR / n).is_file()]
if missing:
    raise FileNotFoundError(f"Thiếu {missing}; chạy code/setup_data.py trước")
hashes = {n: hashlib.sha256((LABELS_DIR / n).read_bytes()).hexdigest() for n in required}
_verify_provenance(hashes, 0)
train_df, val_df, test_df = dataset.load_split(LABELS_DIR, fold=0)
labels_df = pd.read_csv(LABELS_DIR / "labels.csv")
splits = {"train": train_df, "val": val_df, "test": test_df}
report = dataset.check_split(train_df, val_df, test_df, IMAGES_DIR, reference_df=labels_df)
write_json(TABLE_DIR / "split_check.json", report)
write_json(TABLE_DIR / "csv_hashes.json", hashes)

class_map = labels_df[["Label", "Species"]].drop_duplicates().sort_values("Label")
assert class_map["Label"].tolist() == list(range(9))
CLASS_NAMES = class_map["Species"].tolist()
split_table = pd.DataFrame({
    "split": list(splits),
    "images": [len(df) for df in splits.values()],
    "ratio_percent": [len(df) / report["union"] * 100 for df in splits.values()],
})
split_table.to_csv(TABLE_DIR / "split_sizes.csv", index=False)
display(split_table)
print("Overlap:", report["overlap"], "| union:", report["union"], "| missing:", report["missing"])
discrepancies = pd.DataFrame(report["reference_discrepancies"])
discrepancies.to_csv(TABLE_DIR / "source_discrepancies.csv", index=False)
display(discrepancies if not discrepancies.empty else "Không có sai khác nhãn nguồn")

## 3. Phân bố lớp và đối chiếu số liệu tham khảo

In [ ]:
counts = pd.DataFrame({
    name: df["Label"].value_counts().reindex(range(9), fill_value=0).astype(int)
    for name, df in splits.items()
})
counts.index.name = "Label"
counts["all_splits"] = counts[["train", "val", "test"]].sum(axis=1)
counts["labels_csv"] = labels_df["Label"].value_counts().reindex(range(9), fill_value=0).astype(int)
# Table 1 được trích trong README gốc; đây là số tham khảo, không phải kết quả thí nghiệm.
paper_counts = [1125, 1064, 1031, 1022, 1062, 1009, 1074, 1016, 9106]
counts["paper_reference"] = paper_counts
counts["delta_split_vs_labels_csv"] = counts["all_splits"] - counts["labels_csv"]
counts.insert(0, "Species", CLASS_NAMES)
counts.to_csv(TABLE_DIR / "class_counts.csv")
display(counts)
print("Số tham khảo lấy từ README.md gốc, mục Dataset/Table 1.")

In [ ]:
positions = np.arange(9)
fig, ax = plt.subplots(figsize=(14, 5))
for offset, name, color in [(-0.26, "train", "#2563eb"), (0, "val", "#10b981"), (0.26, "test", "#f59e0b")]:
    bars = ax.bar(positions + offset, counts[name], width=0.25, label=name, color=color)
    ax.bar_label(bars, fontsize=7, padding=2)
ax.set_xticks(positions, CLASS_NAMES, rotation=25, ha="right")
ax.set_ylabel("Number of images")
ax.set_title("DeepWeeds fold 0 — class counts (original split targets)")
ax.legend()
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
fig.savefig(FIG_DIR / "class_distribution.png", dpi=150)
plt.show()
plt.close(fig)

fractions = counts[["train", "val", "test"]].div(split_table.set_index("split")["images"], axis=1)
fig, ax = plt.subplots(figsize=(10, 5))
bottom = np.zeros(3)
colors = plt.get_cmap("tab20").colors
for label in range(9):
    values = fractions.loc[label].to_numpy() * 100
    ax.bar(["train", "val", "test"], values, bottom=bottom, color=colors[label],
           label=CLASS_NAMES[label])
    bottom += values
ax.set_ylabel("Percentage within split")
ax.set_title("Class proportions — metadata only")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
fig.savefig(FIG_DIR / "class_proportions.png", dpi=150)
plt.show()
plt.close(fig)

## 4. Mức mất cân bằng và vì sao dùng macro-F1
Ví dụ dưới đây chỉ tính từ **nhãn train**: một bộ dự đoán luôn trả lớp nhiều nhất.
Đây là baseline lý thuyết để hiểu metric, không phải model đã huấn luyện hay điểm test.

In [ ]:
majority_label = int(counts["train"].idxmax())
imbalance_ratio = float(counts["train"].max() / counts["train"].min())
majority_share = float(counts.loc[majority_label, "train"] / len(train_df))
y_train = train_df["Label"].to_numpy(dtype=int)
majority_pred = np.full(len(train_df), majority_label)
majority_probs = np.zeros((len(train_df), 9), dtype=np.float64)
majority_probs[:, majority_label] = 1
majority_metrics = compute_metrics(y_train, majority_pred, majority_probs)
weight_table = counts[["Species", "train"]].copy()
weight_table["inverse_frequency_weight_mean1"] = losses.class_weights(counts["train"].to_numpy()).numpy()
weight_table.to_csv(TABLE_DIR / "train_class_weights.csv")
display(weight_table)
print(f"Lớp đa số: {CLASS_NAMES[majority_label]} ({majority_share:.2%} của train)")
print(f"Tỉ lệ lớp lớn nhất/nhỏ nhất: {imbalance_ratio:.2f}x")
print(f"Always-majority trên nhãn train: accuracy={majority_metrics['top1']:.4f}, "
      f"macro-F1={majority_metrics['macro_f1']:.4f}")

## 5. Audit trùng nội dung byte giữa các tập
SHA256 được tính từ file JPEG gốc để phát hiện **file giống hệt byte**, kể cả tên khác.
Hash này không chứng minh ảnh không trùng về thị giác khi được mã hóa khác nhau.
Nếu có duplicate, ghi nhận hạn chế; không tự xóa ảnh hoặc thay đổi CSV chia tập.
Không xem ảnh test ở cell này và không dùng kết quả để lựa chọn model.

In [ ]:
CHECK_BYTE_DUPLICATES = True
duplicate_groups = []
cross_split_duplicate_groups = []
if CHECK_BYTE_DUPLICATES:
    digest_groups = defaultdict(list)
    for split_name, df in splits.items():
        for row in df.itertuples(index=False):
            digest = hashlib.sha256()
            with (IMAGES_DIR / row.Filename).open("rb") as stream:
                for chunk in iter(lambda: stream.read(1 << 16), b""):
                    digest.update(chunk)
            digest_groups[digest.hexdigest()].append({
                "Filename": row.Filename, "split": split_name, "Label": int(row.Label)})
    duplicate_groups = [members for members in digest_groups.values() if len(members) > 1]
    cross_split_duplicate_groups = [members for members in duplicate_groups
                                   if len({m["split"] for m in members}) > 1]
duplicate_report = {
    "checked": CHECK_BYTE_DUPLICATES, "hash": "SHA256 of original encoded image bytes",
    "duplicate_groups": duplicate_groups, "cross_split_groups": cross_split_duplicate_groups,
}
write_json(TABLE_DIR / "byte_duplicates.json", duplicate_report)
print("Duplicate groups:", len(duplicate_groups),
      "| groups crossing splits:", len(cross_split_duplicate_groups),
      "| checked:", CHECK_BYTE_DUPLICATES)
if cross_split_duplicate_groups:
    display(pd.DataFrame([row for group in cross_split_duplicate_groups for row in group]))

## 6. Chất lượng ảnh train: kích thước, mode, dung lượng và độ sáng
Mở/giải mã toàn bộ ảnh **train**. Độ sáng là mean của kênh L do Pillow chuyển đổi, chia 255.
Các lỗi ảnh được ghi và dừng notebook, không bỏ mẫu lỗi để tiếp tục như dữ liệu đã hợp lệ.

Thống kê RGB được tính trên **512 ảnh train lấy ngẫu nhiên với seed cố định**. Đây là thống kê
mẫu phục vụ EDA; không dùng pixel val/test và không tự thay mean/std của pretrained weights.

In [ ]:
PIXEL_SAMPLE_SIZE = min(512, len(train_df))
pixel_sample = train_df.sample(n=PIXEL_SAMPLE_SIZE, random_state=SEED)
pixel_sample.to_csv(TABLE_DIR / "pixel_sample_train.csv", index=False)
pixel_sample_names = set(pixel_sample["Filename"])
rgb_hist = np.zeros((3, 256), dtype=np.int64)
metadata_rows, image_errors = [], []
for i, row in enumerate(train_df.itertuples(index=False), 1):
    path = IMAGES_DIR / row.Filename
    try:
        with Image.open(path) as image:
            original_mode = image.mode
            width, height = image.size
            channels = len(image.getbands())
            image.load()
            mean_luma = float(np.asarray(image.convert("L")).mean() / 255)
            if row.Filename in pixel_sample_names:
                rgb = np.asarray(image.convert("RGB"))
                for channel in range(3):
                    rgb_hist[channel] += np.bincount(rgb[:, :, channel].ravel(), minlength=256)
        metadata_rows.append({
            "Filename": row.Filename, "Label": int(row.Label), "Species": row.Species,
            "width": width, "height": height, "original_mode": original_mode, "channels": channels,
            "bytes": path.stat().st_size, "mean_luma_0_1": mean_luma,
        })
    except (OSError, ValueError) as error:
        image_errors.append({"Filename": row.Filename, "error": str(error)})
    if i % 2500 == 0:
        print(f"Audited {i}/{len(train_df)} train images")
image_metadata = pd.DataFrame(metadata_rows)
image_metadata.to_csv(TABLE_DIR / "image_metadata_train.csv", index=False)
write_json(TABLE_DIR / "image_errors_train.json", image_errors)
if image_errors:
    raise RuntimeError(f"{len(image_errors)} ảnh train lỗi; xem image_errors_train.json")
assert len(image_metadata) == len(train_df)
display(image_metadata.groupby(["width", "height", "original_mode", "channels"]).size().rename("images").to_frame())
display(image_metadata[["bytes", "mean_luma_0_1"]].describe())

pixel_values = np.arange(256, dtype=np.float64) / 255
pixel_count = rgb_hist.sum(axis=1)
rgb_mean = (rgb_hist * pixel_values).sum(axis=1) / pixel_count
rgb_std = np.sqrt(np.maximum(0, (rgb_hist * pixel_values**2).sum(axis=1) / pixel_count - rgb_mean**2))
pixel_stats = {"scope": "seeded random TRAIN sample only", "seed": SEED, "images": PIXEL_SAMPLE_SIZE,
               "pixels_per_channel": int(pixel_count[0]),
               "mean_rgb_0_1": rgb_mean.tolist(), "std_rgb_0_1": rgb_std.tolist(),
               "pretrained_normalization_policy": "retain the chosen model pretrained_cfg mean/std"}
write_json(TABLE_DIR / "pixel_stats_train.json", pixel_stats)
display(pd.DataFrame({"channel": ["R", "G", "B"], "sample_mean": rgb_mean, "sample_std": rgb_std,
                      "ImageNet_mean_reference": dataset.IMAGENET_MEAN,
                      "ImageNet_std_reference": dataset.IMAGENET_STD}))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(image_metadata["bytes"] / 1024, bins=40, color="#2563eb")
axes[0].set(xlabel="JPEG size (KiB)", ylabel="Train images", title="File sizes")
axes[1].hist(image_metadata["mean_luma_0_1"], bins=40, color="#10b981")
axes[1].set(xlabel="Mean grayscale intensity (0–1)", ylabel="Train images", title="Brightness proxy")
for channel, color in enumerate(["red", "green", "blue"]):
    axes[2].plot(pixel_values, rgb_hist[channel] / pixel_count[channel], color=color, label="RGB"[channel])
axes[2].set(xlabel="Channel intensity (0–1)", ylabel="Pixel fraction", title=f"RGB distribution — {PIXEL_SAMPLE_SIZE} train images")
axes[2].legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "train_image_statistics.png", dpi=150)
plt.show()
plt.close(fig)

brightness_by_class = image_metadata.groupby("Label")["mean_luma_0_1"].agg(["count", "mean", "std", "min", "max"])
brightness_by_class.insert(0, "Species", CLASS_NAMES)
brightness_by_class.to_csv(TABLE_DIR / "brightness_by_class_train.csv")
display(brightness_by_class)

## 7. Xem ít nhất 3 ảnh mỗi lớp — chỉ lấy từ train
Mỗi trang chứa 3 lớp × 3 ảnh gốc. Tên file được lưu để có thể xem lại cùng mẫu.
Quan sát hình lá, thân, màu hoa/quả, nền đất/cỏ và vật thể nhỏ. Đặc biệt xem Chinee apple ↔ Snake weed
và độ đa dạng của lớp Negative. Nhận xét trực quan là giả thuyết; chưa có dự đoán model để xác nhận lỗi.

In [ ]:
sample_parts = []
for label in range(9):
    selected = train_df.loc[train_df["Label"] == label].sample(n=3, random_state=SEED + label)
    sample_parts.append(selected)
visual_samples = pd.concat(sample_parts, ignore_index=True)
visual_samples.to_csv(TABLE_DIR / "visual_samples_train.csv", index=False)
for page in range(3):
    fig, axes = plt.subplots(3, 3, figsize=(11, 10))
    for local_row, label in enumerate(range(page * 3, page * 3 + 3)):
        selected = visual_samples.loc[visual_samples["Label"] == label]
        for column, row in enumerate(selected.itertuples(index=False)):
            with Image.open(IMAGES_DIR / row.Filename) as image:
                axes[local_row, column].imshow(image.convert("RGB"))
            axes[local_row, column].set_title(f"{CLASS_NAMES[label]}\n{row.Filename}", fontsize=9)
            axes[local_row, column].axis("off")
    fig.suptitle(f"TRAIN original images — classes {page * 3}–{page * 3 + 2}")
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"train_samples_{page + 1:02d}.png", dpi=130)
    plt.show()
    plt.close(fig)

## 8. Kiểm tra augmentation và giải chuẩn hóa
So sánh ảnh gốc với val deterministic và các biến thể train. Dùng normalize mặc định ImageNet ở cell
này; khi training, lấy mean/std từ pretrained_cfg của model đã chọn. Kiểm tra crop có bỏ mất cây mục tiêu,
đổi màu có quá mạnh và nhãn có còn hợp lý không.

In [ ]:
def denormalize(tensor):
    mean = torch.tensor(dataset.IMAGENET_MEAN)[:, None, None]
    std = torch.tensor(dataset.IMAGENET_STD)[:, None, None]
    return (tensor.detach().cpu() * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

preview_labels = [0, 7, 8]
preview_rows = [visual_samples.loc[visual_samples["Label"] == label].iloc[0] for label in preview_labels]
transforms_to_show = [
    ("Val center crop", dataset.build_transforms(False, 224)),
    ("Train basic", dataset.build_transforms(True, 224, "basic")),
    ("Train color", dataset.build_transforms(True, 224, "color")),
    ("Train trivial", dataset.build_transforms(True, 224, "trivial")),
    ("Train randaug", dataset.build_transforms(True, 224, "randaug")),
]
fig, axes = plt.subplots(3, 6, figsize=(18, 9))
for row_index, row in enumerate(preview_rows):
    with Image.open(IMAGES_DIR / row["Filename"]) as image:
        image = image.convert("RGB")
        axes[row_index, 0].imshow(image)
        axes[row_index, 0].set_title(f"Original: {CLASS_NAMES[int(row['Label'])]}", fontsize=9)
        for column, (name, transform) in enumerate(transforms_to_show, 1):
            random.seed(SEED + row_index)
            np.random.seed(SEED + row_index)
            torch.manual_seed(SEED + row_index)
            axes[row_index, column].imshow(denormalize(transform(image)))
            axes[row_index, column].set_title(name, fontsize=9)
    for ax in axes[row_index]:
        ax.axis("off")
fig.suptitle("TRAIN preview — denormalized transformations")
fig.tight_layout()
fig.savefig(FIG_DIR / "augmentation_preview.png", dpi=130)
plt.show()
plt.close(fig)

## 9. Kiểm tra Mixup/CutMix: ảnh và nhãn cùng được trộn
Cell minh họa chỉ dùng ảnh train. Lam của CutMix đã được điều chỉnh theo diện tích vùng dán sau clipping.
Ảnh minh họa không thay cho kiểm tra overfit một batch và loss ban đầu ở giai đoạn sanity checks.

In [ ]:
mixed_labels = [0, 1, 7, 8]
mixed_rows = [visual_samples.loc[visual_samples["Label"] == label].iloc[0] for label in mixed_labels]
val_transform = dataset.build_transforms(False, 224)
images = []
for row in mixed_rows:
    with Image.open(IMAGES_DIR / row["Filename"]) as image:
        images.append(val_transform(image.convert("RGB")))
x = torch.stack(images)
y = torch.tensor(mixed_labels)
mixed_batches = []
for method in ("mixup", "cutmix"):
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    mixed_x, (y_a, y_b, lam) = losses.mix_batch(x, y, alpha=1.0, mode=method)
    mixed_batches.append((method, mixed_x, y_a, y_b, lam))
fig, axes = plt.subplots(3, 4, figsize=(14, 10))
for column in range(4):
    axes[0, column].imshow(denormalize(x[column]))
    axes[0, column].set_title(f"Original: {CLASS_NAMES[int(y[column])]}", fontsize=9)
for row_index, (method, mixed_x, y_a, y_b, lam) in enumerate(mixed_batches, 1):
    for column in range(4):
        axes[row_index, column].imshow(denormalize(mixed_x[column]))
        axes[row_index, column].set_title(
            f"{method}, lam={lam:.3f}\n{CLASS_NAMES[int(y_a[column])]} + {CLASS_NAMES[int(y_b[column])]}", fontsize=9)
for ax in axes.flat:
    ax.axis("off")
fig.suptitle("TRAIN batch mixing — image and target pairs")
fig.tight_layout()
fig.savefig(FIG_DIR / "mixup_cutmix_preview.png", dpi=130)
plt.show()
plt.close(fig)

## 10. Tổng hợp EDA và bước tiếp theo
Các ghi chú tự động dưới đây chỉ mô tả số đo/audit. Bổ sung nhận xét ảnh bằng mắt trước khi chọn
augmentation. Sau EDA: kiểm tra CE ban đầu, overfit một batch train, rồi chạy baseline/backbone.
Không chọn cấu hình từ test và không điều chỉnh split theo kết quả audit.

In [ ]:
eda_summary = {
    "fold": 0, "seed": SEED, "split_sizes": report["n"], "overlap": report["overlap"],
    "union": report["union"], "missing_images": report["missing"],
    "majority_class_train": CLASS_NAMES[majority_label],
    "majority_share_train": majority_share, "imbalance_ratio_train": imbalance_ratio,
    "train_label_only_majority_top1": float(majority_metrics["top1"]),
    "train_label_only_majority_macro_f1": float(majority_metrics["macro_f1"]),
    "train_images_decoded": len(image_metadata), "image_error_count": len(image_errors),
    "train_resolution_modes": image_metadata.groupby(["width", "height", "original_mode"]).size()
        .reset_index(name="images").to_dict("records"),
    "pixel_statistics": pixel_stats,
    "byte_duplicate_check_completed": CHECK_BYTE_DUPLICATES,
    "byte_duplicate_groups": len(duplicate_groups),
    "cross_split_byte_duplicate_groups": len(cross_split_duplicate_groups),
    "source_discrepancies": report["reference_discrepancies"],
    "visual_samples_per_class": 3, "visual_images_split": "train",
}
write_json(TABLE_DIR / "eda_summary.json", eda_summary)
notes = [
    "# Ghi chú EDA DeepWeeds — fold 0",
    "",
    f"- Train/val/test: {len(train_df):,}/{len(val_df):,}/{len(test_df):,}; union {report['union']:,}.",
    f"- Ba giao: {report['overlap']}; không thiếu ảnh trong CSV.",
    f"- Lớp đa số train: {CLASS_NAMES[majority_label]}, chiếm {majority_share:.2%}; imbalance {imbalance_ratio:.2f}x.",
    f"- Baseline lý thuyết luôn đoán lớp đa số trên nhãn TRAIN: top-1 {majority_metrics['top1']:.4f}, macro-F1 {majority_metrics['macro_f1']:.4f}.",
    f"- Đã giải mã {len(image_metadata):,} ảnh train; lỗi {len(image_errors)}.",
    f"- Thống kê RGB lấy từ {PIXEL_SAMPLE_SIZE} ảnh train với seed {SEED}; giữ normalize của pretrained config.",
    f"- Duplicate byte groups: {len(duplicate_groups)}; cross-split: {len(cross_split_duplicate_groups)}; checked={CHECK_BYTE_DUPLICATES}.",
    f"- Sai khác nhãn giữa nguồn split/labels.csv: {len(report['reference_discrepancies'])}; giữ split nguyên bản.",
    "- Ảnh mẫu mỗi lớp và ảnh augmentation/Mixup/CutMix nằm trong curves/eda/.",
    "",
    "## Câu hỏi nhận xét trực quan cần bổ sung",
    "",
    "- Cặp Chinee apple/Snake weed: chi tiết nào giúp phân biệt? Có crop làm mất chi tiết đó không?",
    "- Negative gồm những kiểu nền/đối tượng nào trong các mẫu đã xem?",
    "- Color/RandAugment/Mixup/CutMix giữ được dấu hiệu loài trong mẫu nào, phá hủy dấu hiệu trong mẫu nào?",
    "",
    "Các nhận xét ảnh là giả thuyết; cần ablation trên val để kiểm chứng. Không dùng điểm test cho lựa chọn.",
]
(TABLE_DIR / "eda_notes.md").write_text("\n".join(notes) + "\n", encoding="utf-8")
display(eda_summary)
print("Figures:", FIG_DIR)
print("Tables/summary:", TABLE_DIR)